# Снимок ноутбука RTMDet-s для этого опыта

Исходники обучения: commit `61199ac4ebc09baa9ce15f6cd9e7e6de2c1947ac` ветки `pipeline`.
Пакеты обучения пока находятся в той ветке; этот отчёт в `main` их не переносит.
См. `reproduce.md`: сначала подготовьте закреплённый checkout и окружение, затем задайте
`CERAMIC_SOURCE_ROOT` (путь к нему) и `CERAMIC_IMAGES` (путь к фото).

Тяжёлые выводы удалены; результаты сохранены рядом в JSON/CSV/PNG.
В исходном коде изменён только поиск корня проекта: добавлена переменная `CERAMIC_SOURCE_ROOT`.
Открытие ноутбука само по себе не запускает обучение.


# RTMDet-s: обучение детекции клейм

Обучение официального RTMDet-s из MMDetection через универсальный пайплайн CeramicNet.
Текущий протокол **v3_seed42_reserve80_active15**: **259 train** (203 stamps + 56 негативов)
и **145 val** (51 stamps + 94 негатива), seed 42.

Из каждого источника `empty` и `ornaments_without_copies` сначала откладываются 20%
кадров в val. Из объединённых оставшихся 80% (374 фото) берутся 15% (56 фото) для
фактического train. Остальные **318 кадров резерва не используются** ни в train, ни в val.
Всё уже записано в JSON; повторного отбора в этом ноутбуке нет.

`dataset.StampDataset` проверяет данные и показывает примеры. Штатный `CocoDataset`
MMDetection читает те же train/val JSON; `TeamAugment` подключает аугментации команды.
Фон задаётся пустыми annotations, отдельного класса «фон» нет.

Обучение: исходный COCO pretrained detector → один класс `kleimo`, AdamW, EMA,
лучший checkpoint по val mAP@[.50:.95]. Старые веса с другого split не используются.
Параметры и аугментации берутся из `configs/experiments.json`: по умолчанию 50 эпох, 640.

В v3 есть 41 пересечение предметов между фактическими train/val (разные снимки),
включая 8 пересечений между негативами. Точных копий и общих файлов нет.
Списки записаны в manifest. Val используется для выбора весов и порога;
независимого test нет, оценка не подтверждает обобщение на полностью новые предметы.

## 1. Окружение и локальные папки

На Mac из корня CeramicNet один раз выполнить с **Python 3.12**:

```bash
python3.12 train_pipelines/detection_models/RTMDet-s/setup_local.py
```

Выбрать ядро **CeramicNet RTMDet (CPU)**. Скрипт создаёт отдельное `.venv-rtmdet`,
собирает полноценный MMCV 2.1.0 с CPU-операторами и устанавливает MMDetection 3.3.0.
Первая сборка может занять несколько минут; нужен установленный Apple Command Line Tools.
`mmcv-lite` не подходит для запуска детектора.

Ожидаемая структура фотографий рядом с CeramicNet:

```text
Клеймы/
  stamps/
  empty/
  ornaments_without_copies/
```

Если папка другая, задать `CERAMIC_IMAGES` или изменить `IMAGES` ниже.
На Linux/CUDA нужны совместимая пара torch/torchvision и полноценный MMCV;
сам ноутбук использует CUDA при её наличии. На Mac явно выбран CPU, поскольку
у данной сборки MMCV нет MPS-операторов. Установка описана в README этой папки.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import json
import os
import subprocess
import sys
import urllib.request

# Работает при запуске ноутбука из его папки или из корня репозитория.
SOURCE_CANDIDATES = [Path(os.environ["CERAMIC_SOURCE_ROOT"]).expanduser().resolve()] if os.environ.get("CERAMIC_SOURCE_ROOT") else [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in SOURCE_CANDIDATES
                if (p / "detection_pipeline").is_dir() and (p / "dataset").is_dir()), None)
if PROJECT is None:
    raise RuntimeError("Открой ноутбук внутри клона CeramicNet из ветки pipeline")
sys.path.insert(0, str(PROJECT))
FOLDER = PROJECT / "train_pipelines/detection_models/RTMDet-s"
IMAGES = Path(os.environ.get("CERAMIC_IMAGES", str(PROJECT.parent / "Клеймы"))).expanduser().resolve()
SPLITS = PROJECT / "detection_pipeline/splits/v3_seed42_reserve80_active15"
MMDET_REPO = Path(os.environ.get("MMDET_REPO", str(PROJECT / ".local/rtmdet/mmdetection"))).expanduser().resolve()
EXPERIMENTS_PATH = PROJECT / "configs/experiments.json"
EXPERIMENTS = json.loads(EXPERIMENTS_PATH.read_text())

# Для первого локального прогона можно поставить EPOCHS=1–3; полный опыт — 50.
EPOCHS = int(os.environ.get("CERAMIC_RTMDET_EPOCHS", EXPERIMENTS["epochs"]))
SEED = int(EXPERIMENTS["seed"])
TRACKING = "local"  # clearml — после отдельной настройки SDK и учётной записи

import torch
import mmcv
import mmengine
import mmdet
from mmcv.ops import nms

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = int(os.environ.get("CERAMIC_RTMDET_BATCH", EXPERIMENTS["batch_size"] if DEVICE != "cpu" else 2))
THREADS = 4
if DEVICE == "cpu":
    torch.set_num_threads(THREADS)
if EPOCHS < 1 or BATCH_SIZE < 1:
    raise ValueError("EPOCHS и BATCH_SIZE должны быть положительными")
assert IMAGES.is_dir(), f"Не найден корень фото: {IMAGES}"
assert (MMDET_REPO / "tools/train.py").is_file(), "Сначала установи MMDetection; см. README"
boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 9., 9.]])
_, keep = nms(boxes, torch.tensor([0.9, 0.8]), 0.5)
assert keep.tolist() == [0], "Проверка CPU NMS не пройдена"

RUN_NAME = os.environ.get("CERAMIC_RTMDET_RUN_NAME") or ("rtmdet_s_v3_" + datetime.now(ZoneInfo("Europe/Moscow")).strftime("%Y%m%d_%H%M%S_%f"))
RUN_ROOT = PROJECT / "runs" / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=False)
WORK_DIR = RUN_ROOT / "train"
CONFIG_DIR = RUN_ROOT / "configs"
print("Проект:", PROJECT)
print("Фото:", IMAGES)
print("torch / mmcv / mmengine / mmdet:", torch.__version__, mmcv.__version__, mmengine.__version__, mmdet.__version__)
print("Устройство:", DEVICE, "| эпохи:", EPOCHS, "| batch:", BATCH_SIZE)
print("Результаты:", RUN_ROOT)

## 2. Сохранённый split и StampDataset

Читаем все записи из v2. Внешние негативы к Dataset повторно не добавляем:
они уже находятся в COCO JSON. Проверяем наличие файлов, SHA256 и размеры фото.
Профиль train берётся из общей конфигурации; val без случайных аугментаций.

In [ ]:
from collections import Counter
import numpy as np
from dataset import StampDataset
from dataset.transforms.factory import build_augmentations
from detection_pipeline.data import read_json, tracking_split_files
from detection_pipeline.experiments import resolve_augmentation

manifest = read_json(SPLITS / "manifest.json")
COCO = {part: read_json(SPLITS / f"{part}.json") for part in ("train", "val")}
PROFILE, OPERATIONS = resolve_augmentation("rtmdet_s", experiments=EXPERIMENTS_PATH)
DATASETS = {}
for part, data in COCO.items():
    annotated = {a["image_id"] for a in data["annotations"]}
    negative_count = sum(im["id"] not in annotated for im in data["images"])
    expected = (manifest["stats"][part]["images"], manifest["stats"][part]["negative_images"])
    assert (len(data["images"]), negative_count) == expected
    transform = build_augmentations(operations=OPERATIONS) if part == "train" and PROFILE != "native" else None
    DATASETS[part] = StampDataset(
        IMAGES, ann_file=SPLITS / f"{part}.json", image_root=IMAGES,
        model="rtmdet", negative_ratio=0, transform=transform)
    for record in data["images"]:
        path = IMAGES / record["file_name"]
        assert path.is_file(), f"Нет фотографии: {path}"
        assert hashlib.sha256(path.read_bytes()).hexdigest() == record["sha256"], f"Изменилось фото: {path}"
    # load_raw также проверяет реальные размеры каждого изображения.
    for index in range(len(DATASETS[part])):
        DATASETS[part].load_raw(index)
    print(part, {"фото": len(data["images"]), "bbox": len(data["annotations"]),
                 "негативы": negative_count, "источники": dict(Counter(Path(im["file_name"]).parts[0] for im in data["images"]))})
assert not ({im["sha256"] for im in COCO["train"]["images"]} & {im["sha256"] for im in COCO["val"]["images"]})
print("Профиль аугментаций:", PROFILE, OPERATIONS)
print("Пересечения предметов из manifest:", manifest["negatives"]["group_overlap"])

# Резерв — только протокол отбора и артефакты логирования, не вход модели.
RESERVE = read_json(SPLITS / "negative_train_reserve.json")
UNUSED = read_json(SPLITS / "negative_unused.json")
reserve_hashes = {im["sha256"] for im in RESERVE["images"]}
unused_hashes = {im["sha256"] for im in UNUSED["images"]}
train_positive_ids = {ann["image_id"] for ann in COCO["train"]["annotations"]}
active_hashes = {im["sha256"] for im in COCO["train"]["images"] if im["id"] not in train_positive_ids}
val_hashes = {im["sha256"] for im in COCO["val"]["images"]}
assert active_hashes <= reserve_hashes
assert not (reserve_hashes & val_hashes)
assert not (active_hashes & unused_hashes)
assert active_hashes | unused_hashes == reserve_hashes
assert not (unused_hashes & {im["sha256"] for im in COCO["train"]["images"]})
for filename, expected in manifest["negatives"]["reserve_files"].items():
    assert hashlib.sha256((SPLITS / filename).read_bytes()).hexdigest() == expected["sha256"]
SPLIT_FILES = tracking_split_files(SPLITS / "train.json", SPLITS / "val.json")
SPLIT_HASHES = {name: hashlib.sha256(Path(path).read_bytes()).hexdigest() for name, path in SPLIT_FILES.items()}
print("Резерв:", len(reserve_hashes), "| активные негативы:", len(active_hashes), "| не используются:", len(unused_hashes))
print("Fingerprint протокола:", manifest["fingerprint"])
print("Identity включает:", list(SPLIT_HASHES))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

val_ds = DATASETS["val"]
positive_index = next(i for i, item in enumerate(val_ds.items) if len(item["boxes"]))
negative_index = next(i for i, item in enumerate(val_ds.items) if not len(item["boxes"]))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, index in zip(axes, [positive_index, negative_index]):
    image, target = val_ds.load_raw(index)
    ax.imshow(image)
    for x1, y1, x2, y2 in target["boxes"]:
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color="lime", linewidth=2))
    ax.set_title(f"{target['file_name']}\nbbox: {len(target['boxes'])}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
fig.savefig(RUN_ROOT / "dataset_examples.png", dpi=140)
plt.show()

## 3. COCO pretrained веса и конфигурация общего пайплайна

Загружаем официальный **RTMDet-s COCO detector**, а не только ImageNet backbone.
Голова меняется с 80 классов на один: сообщение о несовпадении размеров классификатора
при загрузке ожидаемо. Backbone и остальные совместимые параметры переносятся.
Фиксируем URL, SHA256, версии, split и фактический бюджет в metadata.

На CPU уменьшаем только batch и число workers. Размер 640, схема аугментаций,
seed, оценка на полном val и правило выбора checkpoint остаются общими.

In [ ]:
WEIGHTS_URL = (
    "https://download.openmmlab.com/mmdetection/v3.0/rtmdet/"
    "rtmdet_s_8xb32-300e_coco/rtmdet_s_8xb32-300e_coco_20220905_161602-387a891e.pth")
WEIGHTS = PROJECT / ".local/rtmdet/checkpoints/rtmdet_s_coco.pth"
WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
if not WEIGHTS.is_file():
    partial = WEIGHTS.with_suffix(".download")
    try:
        urllib.request.urlretrieve(WEIGHTS_URL, partial)
        partial.replace(WEIGHTS)
    finally:
        partial.unlink(missing_ok=True)
weights_sha = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
assert weights_sha.startswith("387a891e"), "Контрольная сумма не соответствует официальным COCO весам"
print("COCO checkpoint:", WEIGHTS, "| SHA256:", weights_sha)

In [ ]:
from mmengine.config import Config
from detection_pipeline.native_configs import generate
from detection_pipeline.data import write_json

files = generate(SPLITS, IMAGES, CONFIG_DIR, mmdet_repo=MMDET_REPO,
                 epochs=EPOCHS, batch_size=BATCH_SIZE, experiments=EXPERIMENTS_PATH)
base_config = next(Path(p) for p in files if Path(p).name.startswith("rtmdet_s_"))
cfg = Config.fromfile(str(base_config))
cfg.work_dir = str(WORK_DIR)
cfg.load_from = str(WEIGHTS)
cfg.resume = False  # Новый split: обучение с исходных COCO-весов
cfg.model.backbone.init_cfg = None  # load_from уже содержит backbone
cfg.launcher = "none"
if DEVICE == "cpu":
    cfg.custom_imports.imports.append("detection_pipeline.integrations.cpu_runner")
    cfg.runner_type = "CPURunner"
    cfg.env_cfg.mp_cfg = dict(mp_start_method="spawn", opencv_num_threads=0)
    cfg.env_cfg.dist_cfg.backend = "gloo"
for part in ("train", "val", "test"):
    loader = cfg[f"{part}_dataloader"]
    loader.num_workers = 0
    loader.persistent_workers = False
    loader.pin_memory = DEVICE != "cpu"
    loader.drop_last = False
    if part != "train":
        loader.batch_size = 1
    loader.dataset.filter_cfg = dict(filter_empty_gt=False, min_size=0)
# Одноэпоховый пробный опыт не должен создавать cosine с begin == end.
cfg.param_scheduler = [s for s in cfg.param_scheduler if s.get("end", EPOCHS) > s.get("begin", 0)]
cfg.default_hooks.logger.interval = 10
cfg.default_hooks.logger.log_metric_by_epoch = True
cfg.log_processor.window_size = 10
CONFIG = CONFIG_DIR / "rtmdet_s.py"
cfg.dump(str(CONFIG))
metadata = read_json(base_config.with_suffix(".metadata.json"))
metadata.update(pretraining="COCO detector; new one-class classification layers", weights_url=WEIGHTS_URL,
                weights_sha256=weights_sha, device=DEVICE, epochs=EPOCHS, batch_size=BATCH_SIZE,
                image_size=640, learning_rate=cfg.optim_wrapper.optimizer.lr,
                split_version=SPLITS.name, split_fingerprint=manifest["fingerprint"],
                split_files_sha256=SPLIT_HASHES, repository_commit=subprocess.check_output(
                    ["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip(), tracking=TRACKING,
                versions={"torch": torch.__version__, "mmcv": mmcv.__version__,
                          "mmengine": mmengine.__version__, "mmdet": mmdet.__version__})
write_json(CONFIG.with_suffix(".metadata.json"), metadata)
print(json.dumps(metadata, indent=2, ensure_ascii=False))
print("Разрешённый конфиг:", CONFIG)

## 4. Проверка настоящих загрузчиков MMDetection

Проверяем, что train и val содержат **все** негативы, а общий `TeamAugment`
зарегистрирован. Чтение этого батча не изменяет сохранённые split.
Новый процесс обучения сам выставит seed из конфига.

In [ ]:
from mmengine.runner import Runner
from mmdet.utils import register_all_modules
register_all_modules(init_default_scope=True)
for part in ("train", "val"):
    expected_negatives = manifest["stats"][part]["negative_images"]
    loader = Runner.build_dataloader(cfg[f"{part}_dataloader"], seed=SEED)
    native_ds = loader.dataset
    indices = [native_ds.get_data_info(i) for i in range(len(native_ds))]
    assert len(native_ds) == manifest["stats"][part]["images"]
    assert {r["img_id"] for r in indices} == {r["id"] for r in COCO[part]["images"]}
    assert sum(not r["instances"] for r in indices) == expected_negatives
    batch = next(iter(loader))
    print(part, "фото:", len(native_ds), "негативы:", expected_negatives,
          "первый батч:", [tuple(x.shape) for x in batch["inputs"]])
    del loader, native_ds, batch

## 5. Обучение

Ячейка запускает обучение на всём train и валидацию после каждой эпохи.
Для короткой локальной проверки задайте `EPOCHS=1–3` в начале и выполните ноутбук заново.
Полный опыт использует 50 эпох. Каждый запуск создаёт новую папку `runs/rtmdet_s_*`.

Используется общая обёртка `detection_pipeline.train_native`, поэтому параметры,
хеши train/val, loss, lr и val mAP сохраняются общим логгером. По умолчанию `local`;
для сервера ClearML сначала установите `requirements-tracking.txt`, настройте
`clearml-init` и смените `TRACKING` в начале. Ключи в ноутбук не вставляются.

In [ ]:
# Поток вывода краткий; полный журнал сохраняется рядом с результатами.
def run_logged(command, logfile):
    env = {**os.environ, "OMP_NUM_THREADS": str(THREADS), "MKL_NUM_THREADS": str(THREADS),
           "PYTHONUNBUFFERED": "1", "MPLBACKEND": "Agg"}
    process = subprocess.Popen([str(x) for x in command], cwd=PROJECT, env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        with Path(logfile).open("w", encoding="utf-8") as log:
            for line in process.stdout:
                log.write(line)
                log.flush()
                if any(token in line for token in ("Epoch(", "best checkpoint", "Traceback", "Error:", "CPU selected")):
                    print(line, end="", flush=True)
        status = process.wait()
        if status:
            tail = Path(logfile).read_text().splitlines()[-35:]
            raise RuntimeError("Обучение/оценка завершились с ошибкой:\n" + "\n".join(tail))
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()

run_logged([sys.executable, "-m", "detection_pipeline.train_native", "mmdet",
            "--repo", MMDET_REPO, "--config", CONFIG, "--tracking", TRACKING,
            "--run-name", RUN_NAME], RUN_ROOT / "training.log")
checkpoints = list(WORK_DIR.glob("best_coco_bbox_mAP_*.pth"))
assert len(checkpoints) == 1, f"Ожидался один лучший checkpoint: {checkpoints}"
BEST_CHECKPOINT = checkpoints[0]
print("\nЛучшие веса:", BEST_CHECKPOINT)

## 6. Графики обучения и общий оценщик

Экспортируем лучший checkpoint на **все 145 val фото** с тем же детерминированным
resize/pad и переводом рамок в исходные координаты. Общий оценщик вычисляет mAP,
AP50/AP75, precision/recall и долю ложных срабатываний на 94 негативах.
Дополнительно подбирается порог на val при precision ≥ 90%; если такого порога нет,
`selected = null`. На будущем независимом test порог не подбирается заново.

In [ ]:
logfile = next(WORK_DIR.glob("**/vis_data/scalars.json"))
records = [json.loads(line) for line in logfile.read_text().splitlines() if line.strip()]
train_records = [r for r in records if "loss" in r]
val_records = [r for r in records if "coco/bbox_mAP" in r]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([r["step"] for r in train_records], [r["loss"] for r in train_records], label="train loss")
axes[0].set(xlabel="Итерация", ylabel="Loss")
axes[0].legend()
for key in ["coco/bbox_mAP", "coco/bbox_mAP_50", "coco/bbox_mAP_75"]:
    axes[1].plot([r["step"] for r in val_records], [r[key] for r in val_records], marker="o", label=key.removeprefix("coco/bbox_"))
axes[1].set(xlabel="Эпоха", ylabel="AP", ylim=(0, 1))
axes[1].legend()
plt.tight_layout()
fig.savefig(RUN_ROOT / "learning_curves.png", dpi=160)
plt.show()

In [ ]:
EVAL_DIR = RUN_ROOT / "val"
run_logged([sys.executable, "-m", "detection_pipeline.predict_native", "mmdet",
            "--repo", MMDET_REPO, "--config", CONFIG, "--checkpoint", BEST_CHECKPOINT,
            "--images", IMAGES, "--annotations", SPLITS / "val.json",
            "--output", EVAL_DIR, "--device", DEVICE, "--tune-threshold"], RUN_ROOT / "evaluation.log")
metrics = read_json(EVAL_DIR / "metrics.json")
selected = metrics["threshold_search_on_validation"]["selected"]
print("COCO:", json.dumps(metrics["coco"], indent=2))
print("При threshold=0.5:", json.dumps(metrics["operating_point"], indent=2))
print("Порог с precision >= 90% на val:", selected)
print("Общий логгер:", read_json(WORK_DIR / "tracking/run.json")["status"])
print("Файл метрик:", EVAL_DIR / "metrics.json")

## 7. Предсказания: stamps и негативные фото

Зелёные рамки — разметка, красные — предсказания выше выбранного на val порога
(или 0.5, если ограничение precision не достигнуто). Фото выбраны с seed 42.
На негативных кадрах зелёных рамок нет; красная рамка означает ложное срабатывание.

In [ ]:
artifact = read_json(EVAL_DIR / "predictions.json")
predictions = {r["image_id"]: r["detections"] for r in artifact["records"]}
truth = {}
for ann in COCO["val"]["annotations"]:
    truth.setdefault(ann["image_id"], []).append(ann["bbox"])
rng = np.random.RandomState(SEED)
positives = [im for im in COCO["val"]["images"] if im["id"] in truth]
negatives = [im for im in COCO["val"]["images"] if im["id"] not in truth]
picked = [positives[i] for i in rng.choice(len(positives), 4, replace=False)]
picked += [negatives[i] for i in rng.choice(len(negatives), 2, replace=False)]
threshold = selected["threshold"] if selected else 0.5
from PIL import Image
fig, axes = plt.subplots(3, 2, figsize=(12, 15))
for ax, record in zip(axes.flat, picked):
    with Image.open(IMAGES / record["file_name"]) as im:
        ax.imshow(im.convert("RGB"))
    for x, y, w, h in truth.get(record["id"], []):
        ax.add_patch(Rectangle((x, y), w, h, fill=False, color="lime", linewidth=2))
    detections = [d for d in predictions[record["id"]] if d["score"] >= threshold]
    for d in detections:
        x, y, w, h = d["bbox"]
        ax.add_patch(Rectangle((x, y), w, h, fill=False, color="red", linewidth=2))
        ax.text(x, y, f"{d['score']:.2f}", color="red", bbox=dict(facecolor="white", alpha=0.7))
    ax.set_title(f"{record['file_name']}\n{'stamps' if record['id'] in truth else 'негатив'}; предсказаний: {len(detections)}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
fig.savefig(RUN_ROOT / "val_predictions.png", dpi=160)
plt.show()
print("Порог отображения:", threshold)
print("Все результаты:", RUN_ROOT)

## Результаты запуска

- `train/best_coco_bbox_mAP_*.pth` — выбранные веса.
- `configs/rtmdet_s.py` и `.metadata.json` — фактические параметры, источники весов и версии.
- `train/tracking/` — общий логгер CeramicNet: хеши split, параметры и метрики.
- `val/predictions.json`, `val/metrics.json` — общий формат для сравнения моделей.
- `dataset_examples.png`, `learning_curves.png`, `val_predictions.png` — иллюстрации.
- `training.log`, `evaluation.log` — полные журналы.

Объединить метрики с другими моделями можно через `detection_pipeline.compare`.
Веса и фотографии не включаются в Git. Короткий пробный прогон подтверждает запуск
цикла, но не заменяет полный опыт и независимую оценку качества.